# Handwritten Digit Recognizer (MNIST + CNN)

This notebook trains a Convolutional Neural Network (CNN) to recognize handwritten digits (0-9) using the famous **MNIST** dataset. Every important line has a comment explaining what it does, since this is written for beginners.

**How to use this notebook:** Run each cell from top to bottom, in order (click the play button on the left of each cell, or press `Shift + Enter`).

In [ ]:
# --- Cell 1: Import all the libraries we need ---

# TensorFlow is the main deep learning library we use to build and train the neural network
import tensorflow as tf

# 'layers' and 'models' are shortcuts from Keras (part of TensorFlow) used to build the CNN
from tensorflow.keras import layers, models

# NumPy helps us work with arrays of numbers (images are just arrays of pixel values)
import numpy as np

# Matplotlib is used to draw/plot the digit images so we can see them
import matplotlib.pyplot as plt

# Print the TensorFlow version so we know which version we're using
print("TensorFlow version:", tf.__version__)


In [ ]:
# --- Cell 2: Load the MNIST dataset ---

# MNIST is a dataset of 70,000 grayscale images of handwritten digits (0-9), each 28x28 pixels
# Keras already includes this dataset, so we can load it with one line
mnist = tf.keras.datasets.mnist

# load_data() splits the dataset into training data and testing data automatically
# x_train / x_test = the actual images (pixel data)
# y_train / y_test = the correct labels (the digit each image represents, 0-9)
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Print the shape (dimensions) of the training data to understand what we're working with
# This should show (60000, 28, 28) -> 60,000 images, each 28x28 pixels
print("Training data shape:", x_train.shape)

# Print the shape of the test data
# This should show (10000, 28, 28) -> 10,000 images, each 28x28 pixels
print("Testing data shape:", x_test.shape)


In [ ]:
# --- Cell 3: Preprocess the data (normalize and reshape) ---

# Each pixel value currently ranges from 0 (black) to 255 (white)
# Neural networks train better when input values are small, so we scale them to a range of 0-1
# We do this by dividing every pixel value by 255.0
x_train = x_train / 255.0
x_test = x_test / 255.0

# Our CNN expects images with a "channel" dimension (like RGB images have 3 channels)
# Since MNIST images are grayscale, they only have 1 channel
# We reshape the data from (num_images, 28, 28) to (num_images, 28, 28, 1)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# Print the new shape to confirm the extra "channel" dimension was added
print("New training data shape:", x_train.shape)
print("New testing data shape:", x_test.shape)


In [ ]:
# --- Cell 4: Build the CNN (Convolutional Neural Network) model ---

# 'Sequential' means we stack layers one after another, in a simple straight line
model = models.Sequential([

    # First convolutional layer: scans the image with 32 filters (each 3x3 pixels)
    # to detect simple patterns like edges and curves
    # input_shape=(28, 28, 1) tells the model the shape of a single input image
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),

    # First pooling layer: shrinks the image size by taking the maximum value in each 2x2 block
    # This reduces computation and helps the model focus on the most important features
    layers.MaxPooling2D((2, 2)),

    # Second convolutional layer: uses 64 filters to detect more complex patterns,
    # built on top of the simple patterns found by the first layer
    layers.Conv2D(64, (3, 3), activation='relu'),

    # Second pooling layer: shrinks the image size again
    layers.MaxPooling2D((2, 2)),

    # Third convolutional layer: detects even more complex/abstract patterns
    layers.Conv2D(64, (3, 3), activation='relu'),

    # Flatten layer: converts the 2D grid of features into a single 1D list of numbers
    # so it can be fed into regular (Dense) neural network layers
    layers.Flatten(),

    # Dense (fully connected) layer with 64 neurons that combines all the detected features
    # to start making a decision about which digit this is
    layers.Dense(64, activation='relu'),

    # Final output layer with 10 neurons, one for each possible digit (0-9)
    # 'softmax' converts the outputs into probabilities that add up to 1
    # (e.g. 90% chance it's a 3, 5% chance it's an 8, etc.)
    layers.Dense(10, activation='softmax')
])

# Print a summary of the model: shows every layer, its output shape, and number of parameters
model.summary()


In [ ]:
# --- Cell 5: Compile the model (configure it for training) ---

# compile() sets up how the model will learn
model.compile(
    # 'adam' is a popular optimizer algorithm that adjusts the model's internal
    # numbers (weights) to reduce errors during training
    optimizer='adam',

    # This loss function is used because our labels (y_train, y_test) are plain integers (0-9)
    # rather than one-hot encoded vectors; it measures how wrong the model's predictions are
    loss='sparse_categorical_crossentropy',

    # We want to track accuracy (the percentage of correct predictions) during training
    metrics=['accuracy']
)


In [ ]:
# --- Cell 6: Train the model ---

# fit() trains the model on the training data
history = model.fit(
    x_train, y_train,          # the training images and their correct labels

    epochs=10,                 # number of times the model will go through the entire training dataset

    # validation_split=0.1 means 10% of the training data is set aside (not used for training)
    # and used instead to check how well the model performs on data it hasn't directly learned from
    validation_split=0.1
)


In [ ]:
# --- Cell 7: Evaluate the model on the test dataset ---

# evaluate() checks how well the trained model performs on completely unseen test data
# It returns the loss (error) and the accuracy
test_loss, test_accuracy = model.evaluate(x_test, y_test)

# Print the final test accuracy, formatted as a percentage with 2 decimal places
print(f"\nFinal Test Accuracy: {test_accuracy * 100:.2f}%")


In [ ]:
# --- Cell 8: Make predictions on some test images ---

# predict() runs the model on the test images and returns a probability for each digit (0-9)
predictions = model.predict(x_test)

# For each prediction, np.argmax() picks the digit with the highest probability
# This converts the list of 10 probabilities into a single predicted digit
predicted_labels = np.argmax(predictions, axis=1)

# Print the first 5 predicted labels and the first 5 actual labels, just as a quick text check
print("Predicted labels:", predicted_labels[:5])
print("Actual labels:   ", y_test[:5])


In [ ]:
# --- Cell 9: Display sample images alongside their predicted digits ---

# Set how many sample predictions we want to show
num_samples = 5

# Create a figure (a blank canvas) sized to comfortably fit all sample images side by side
plt.figure(figsize=(10, 3))

# Loop through the first 'num_samples' test images
for i in range(num_samples):

    # Create a subplot: 1 row, 'num_samples' columns, position i+1
    plt.subplot(1, num_samples, i + 1)

    # Display the image; we use reshape(28, 28) to remove the extra "channel" dimension
    # and cmap='gray' to show it as a grayscale image
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')

    # Set the title of this subplot to show both the predicted and actual digit
    plt.title(f"Pred: {predicted_labels[i]}\nActual: {y_test[i]}")

    # Turn off the x/y axis ticks/numbers since they aren't needed for viewing an image
    plt.axis('off')

# Adjust spacing so the images and titles don't overlap
plt.tight_layout()

# Actually render/show the figure with all the sample images
plt.show()


## Done!
You've trained a CNN on MNIST, checked its test accuracy, and visualized some of its predictions. 

**Ideas to try next:**
- Increase `epochs` to see if accuracy improves further.
- Add a `Dropout` layer to help prevent overfitting.
- Try drawing your own digit in an image editor and see if the model can recognize it!